In [31]:
!pip install google-colab-selenium


In [32]:
# Instalación de librerías necesarias para Colab
!pip install requests beautifulsoup4 selenium pandas scrapy google-colab-selenium

In [33]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import google_colab_selenium as gs
import scrapy

# URL de búsqueda en Alkosto (página de listado)
URL_ALKOSTO = "https://www.alkosto.com/search?text=celulares"

# Headers para simular navegador (útil para requests)
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "es-CO,es;q=0.9,en;q=0.8"
}

print("Entorno configurado. URL:", URL_ALKOSTO)

Entorno configurado. URL: https://www.alkosto.com/search?text=celulares


In [34]:
print(" BeautifulSoup con requests (contenido estático) ")

try:
    response = requests.get(URL_ALKOSTO, headers=HEADERS, timeout=15)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, "html.parser")

    # Intentamos con las clases reales de Alkosto
    productos = soup.select("div.product__item")
    print(f"Status: {response.status_code} | Longitud HTML: {len(response.text)} caracteres")
    print(f"Productos encontrados con 'div.product__item': {len(productos)}")

    if len(productos) == 0:
        print("\n CONFIRMADO: los productos NO están en el HTML estático.")
        print("Esto ocurre porque Alkosto los carga dinámicamente con JavaScript.")
        print("Por esta razón se requiere Selenium para renderizar la página antes de parsear.")
    else:
        for i, prod in enumerate(productos[:3], 1):
            nombre = prod.select_one(".product-info--name")
            precio = prod.select_one(".product-info--price-new")
            print(f"Producto {i}: {nombre.get_text(strip=True) if nombre else 'N/A'} | ${precio.get_text(strip=True) if precio else 'N/A'}")

except Exception as e:
    print(f"Error: {e}")

 BeautifulSoup con requests (contenido estático) 
Status: 200 | Longitud HTML: 1730036 caracteres
Productos encontrados con 'div.product__item': 25
Producto 1: N/A | $N/A
Producto 2: N/A | $N/A
Producto 3: N/A | $N/A


In [35]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
import google_colab_selenium as gs

print(" Selenium + BeautifulSoup ")

productos_extraidos = []

try:
    driver = gs.Chrome()
    driver.get(URL_ALKOSTO)

    wait = WebDriverWait(driver, 25)
    wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".js-product-item")))

    soup = BeautifulSoup(driver.page_source, "html.parser")
    items = soup.select(".js-product-item")

    print(f"Productos encontrados: {len(items)}")

    for i, prod in enumerate(items[:10], 1):
        # Nombre (ya funciona)
        nombre_elem = prod.select_one(".js-algolia-product-title, .product-info--name")
        nombre = nombre_elem.get_text(strip=True) if nombre_elem else "Nombre no encontrado"

        # Precio actual: selectores más específicos (evitan el contenedor grande)
        precio_elem = prod.select_one(".product__price--discounts__price, .js-priceProductA")
        if precio_elem:
            precio = precio_elem.get_text(strip=True).split("$")[1]  # Toma solo el primer precio
            precio = "$" + precio if precio else "Precio no encontrado"
        else:
            # Fallback: buscar el primer precio en el contenedor de precio
            precio_container = prod.select_one(".product__item__information__price-section--container")
            precio = precio_container.get_text(strip=True)[:20] if precio_container else "Precio no encontrado"

        # Precio anterior (ya funciona)
        precio_old_elem = prod.select_one(".product__price--discounts__old, .js-priceOldProductA")
        precio_old = precio_old_elem.get_text(strip=True) if precio_old_elem else ""

        productos_extraidos.append({
            "nombre": nombre,
            "precio": precio,
            "precio_anterior": precio_old
        })

        print(f"Producto {i}:")
        print(f"  Nombre: {nombre}")
        print(f"  Precio: {precio}")
        if precio_old:
            print(f"  Precio anterior: {precio_old}")
        print("-" * 40)

    driver.quit()
    print(" Selenium completado.")

except Exception as e:
    print(f"Error en Selenium: {e}")
    try:
        driver.quit()
    except:
        pass

 Selenium + BeautifulSoup 


<IPython.core.display.Javascript object>

Productos encontrados: 25
Producto 1:
  Nombre: Celular SAMSUNG Galaxy A37 256GB 5G Gris
  Precio: $1.649.990
  Precio anterior: $2.299.990
----------------------------------------
Producto 2:
  Nombre: Celular SAMSUNG Galaxy A27 256GB 5G Negro
  Precio: $1.089.050
  Precio anterior: $1.499.050
----------------------------------------
Producto 3:
  Nombre: Celular SAMSUNG Galaxy A27 256GB 5G Azul
  Precio: $1.089.050
  Precio anterior: $1.499.050
----------------------------------------
Producto 4:
  Nombre: Celular OPPO A6 256GB 4G Azul
  Precio: $1.149.900
  Precio anterior: $1.699.900
----------------------------------------
Producto 5:
  Nombre: Celular MOTOROLA G67 256GB 5G Gris
  Precio: $939.050
  Precio anterior: $1.899.050
----------------------------------------
Producto 6:
  Nombre: Celular REDMI Note 17 Pro 256 5G Negro + Power Bank 165W
  Precio: $1.699.050
  Precio anterior: $2.299.050
----------------------------------------
Producto 7:
  Nombre: Celular SAMSUNG Galaxy A

In [36]:
if productos_extraidos:
    df = pd.DataFrame(productos_extraidos)
    print("--- DataFrame de productos extraídos ---")
    print(df.to_string(index=False))
    # Guardar a CSV
    df.to_csv("alkosto_celulares.csv", index=False, encoding="utf-8")
    print("\n Guardado en 'alkosto_celulares.csv'")
else:
    print("No hay datos para mostrar. Revisa la celda anterior.")

--- DataFrame de productos extraídos ---
                                                   nombre     precio precio_anterior
                 Celular SAMSUNG Galaxy A37 256GB 5G Gris $1.649.990      $2.299.990
                Celular SAMSUNG Galaxy A27 256GB 5G Negro $1.089.050      $1.499.050
                 Celular SAMSUNG Galaxy A27 256GB 5G Azul $1.089.050      $1.499.050
                            Celular OPPO A6 256GB 4G Azul $1.149.900      $1.699.900
                       Celular MOTOROLA G67 256GB 5G Gris   $939.050      $1.899.050
 Celular REDMI Note 17 Pro 256 5G Negro + Power Bank 165W $1.699.050      $2.299.050
               Celular SAMSUNG Galaxy A57 256GB 5G Morado $1.999.990      $2.799.990
Celular REDMI Note 17 Pro 256 5G Morado + Power Bank 165W $1.699.050      $2.299.050
                 Celular MOTOROLA G67 256GB 5G Verde Lima   $939.050      $1.899.050
                         Celular HONOR X5d 128GB 4G Negro   $549.050        $899.050

 Guardado en 'alkosto_c

In [37]:
print("--- Spider de Scrapy (definición) ---")

codigo_spider = '''
import scrapy

class AlkostoSpider(scrapy.Spider):
    name = "alkosto_celulares"
    start_urls = ["https://www.alkosto.com/search?text=celulares"]

    def parse(self, response):
        for producto in response.css(".js-product-item"):
            yield {
                "nombre": producto.css(".js-algolia-product-title::text, .product-info--name::text").get(),
                "precio": producto.css(".product__price--discounts__price::text, .js-priceProductA::text").get(),
                "precio_anterior": producto.css(".product__price--discounts__old::text, .js-priceOldProductA::text").get(),
            }
'''

print(codigo_spider)
print("\nNota: Alkosto carga productos con JavaScript, por lo que para ejecutar este")
print("spider en producción se requeriría scrapy-playwright o scrapy-selenium.")
print("Para esta evidencia, se demuestra la estructura del spider y los selectores.")

--- Spider de Scrapy (definición) ---

import scrapy

class AlkostoSpider(scrapy.Spider):
    name = "alkosto_celulares"
    start_urls = ["https://www.alkosto.com/search?text=celulares"]

    def parse(self, response):
        for producto in response.css(".js-product-item"):
            yield {
                "nombre": producto.css(".js-algolia-product-title::text, .product-info--name::text").get(),
                "precio": producto.css(".product__price--discounts__price::text, .js-priceProductA::text").get(),
                "precio_anterior": producto.css(".product__price--discounts__old::text, .js-priceOldProductA::text").get(),
            }


Nota: Alkosto carga productos con JavaScript, por lo que para ejecutar este
spider en producción se requeriría scrapy-playwright o scrapy-selenium.
Para esta evidencia, se demuestra la estructura del spider y los selectores.


In [38]:
print(" Resumen de la práctica ")
print("Herramientas utilizadas: BeautifulSoup, Selenium y Scrapy.")
print("Página analizada: Alkosto (https://www.alkosto.com)")
print("Artículo: Celulares")
print(f"Productos extraídos exitosamente: {len(productos_extraidos)}")
if productos_extraidos:
    print("\nPrimer producto extraído:")
    print(f"  Nombre: {productos_extraidos[0]['nombre']}")
    print(f"  Precio: {productos_extraidos[0]['precio']}")
    print(f"  Precio anterior: {productos_extraidos[0]['precio_anterior']}")
else:
    print("No se extrajeron productos. Revisa los selectores en la Celda 4.")

 Resumen de la práctica 
Herramientas utilizadas: BeautifulSoup, Selenium y Scrapy.
Página analizada: Alkosto (https://www.alkosto.com)
Artículo: Celulares
Productos extraídos exitosamente: 10

Primer producto extraído:
  Nombre: Celular SAMSUNG Galaxy A37 256GB 5G Gris
  Precio: $1.649.990
  Precio anterior: $2.299.990
